In [ ]:
# ==============================================================================
# CELL 0: HARDWARE VERIFICATION & DEPENDENCY SETUP
# ==============================================================================
import subprocess
import sys
import os
import torch

print('=' * 65)
print('  UNSLOTH GEMMA 4 31B LORA TRAINING SETUP (NVIDIA L4 24GB)')
print('=' * 65)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available! Ensure GPU accelerator is enabled.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'Detected GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)')
assert vram_gb >= 20.0, f'Expected >= 20 GB VRAM (L4/A100), found {vram_gb:.2f} GB!'

print('\nInstalling Unsloth, TRL, PEFT, and bitsandbytes with internet enabled...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'torch', 'torchvision', 'torchaudio',
    'bitsandbytes', 'transformers', 'accelerate', 'trl', 'peft'
], check=True)

try:
    import unsloth
    print('Unsloth is already installed.')
except ImportError:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/unslothai/unsloth.git'
    ], check=True)
    import unsloth
    print('Unsloth installed successfully.')


In [ ]:
# ==============================================================================
# CELL 1: DATASET INGESTION & ZERO-LEAK VERIFICATION
# ==============================================================================
import json
from pathlib import Path

WORKING_DIR = Path('/kaggle/working')
DATA_DIR = WORKING_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Discover mounted dataset under /kaggle/input, /kaggle/working/data, or ./data
candidate_train = (
    sorted(Path('/kaggle/input').rglob('unsloth_sft_train.jsonl')) +
    sorted(Path('/kaggle/working/data').rglob('unsloth_sft_train.jsonl')) +
    sorted(Path('./data').rglob('unsloth_sft_train.jsonl'))
)
assert candidate_train, 'Fatal: unsloth_sft_train.jsonl not found in /kaggle/input, /kaggle/working, or ./data!'
train_path = candidate_train[0]
val_path = train_path.parent / 'unsloth_sft_val.jsonl'
assert val_path.exists(), f'Fatal: Missing validation dataset at {val_path}!'

print(f'Using training dataset:   {train_path}')
print(f'Using validation dataset: {val_path}')

# Verify integrity and zero host leaks
bad_markers = ['/private/var/folders', 'swegemma_sandbox', '/Users/yapilymm']
train_tasks = set()
val_tasks = set()

for p, task_set in [(train_path, train_tasks), (val_path, val_tasks)]:
    content = p.read_text(encoding='utf-8')
    lines = [json.loads(line) for line in content.splitlines() if line.strip()]
    leaks = [m for m in bad_markers if m in content]
    print(f'Verified {p.name}: {len(lines)} samples, {len(content)} bytes, host leaks={leaks}')
    assert not leaks, f'Fatal: Leaked host markers {leaks} detected in {p.name}!'
    for row in lines:
        task_set.add(row.get('task_id', ''))
        assert 'prefix_text' in row and 'completion_text' in row and 'text' in row, f'Missing required fields in {p.name} row!'
        assert row.get('target_tools'), f'Dead thought row found in {p.name} (target_tools empty)!'

print(f'Unique task IDs: train={len(train_tasks)} tasks, val={len(val_tasks)} tasks')
assert train_tasks.isdisjoint(val_tasks), f'Fatal: Data leakage detected! Overlap: {train_tasks & val_tasks}'
print('Zero-leakage verification: PASS (train and val task IDs strictly disjoint).')


In [ ]:
# ==============================================================================
# CELL 2: MODEL INITIALIZATION & RANK-8 PEFT CONFIGURATION
# ==============================================================================
import torch
from unsloth import FastLanguageModel
from pathlib import Path

# Max sequence length set to 3072 to strictly fit within 24GB L4 GPU VRAM
max_seq_length = 3072
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f'Discovering and loading Gemma 31B QAT checkpoint (max_seq_length={max_seq_length}, dtype={dtype})...')

candidate_model_dirs = [
    Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2'),
    Path('/kaggle/input/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2'),
    *sorted(Path('/kaggle/input').glob('**/*gemma-4-31b-it-qat-w4a16-ct*')),
    *sorted(Path('models').glob('**/*gemma-4-31b-it-qat-w4a16-ct*')),
]
model_name = None
for cand in candidate_model_dirs:
    if cand.exists() and cand.is_dir():
        model_name = str(cand)
        break
if not model_name:
    model_name = 'google/gemma-4-31b-it-qat-w4a16-ct'
print(f'Using model path: {model_name}')

# CRITICAL GROUND TRUTH: google/gemma-4-31b-it-qat-w4a16-ct is already pre-quantized
# via compressed-tensors. We MUST pass load_in_4bit=False, use_exact_model_name=True,
# and finetune_vision_layers=False to avoid BnB collision or double-quantization errors.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=False,
    use_exact_model_name=True,
    finetune_vision_layers=False,
)

# Target strictly q_proj, v_proj, o_proj (170 modules across 60 layers)
# Omitting k_proj preserves attention_k_eq_v=True on 10 global layers and prevents tool syntax degradation.
# Freezing MLPs prevents catastrophic forgetting and keeps LoRA adapter under 100 MB.
print('Configuring Rank-8 LoRA adapter on attention/output heads (q_proj, v_proj, o_proj)...')
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=['q_proj', 'v_proj', 'o_proj'],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

model.print_trainable_parameters()
print('Model initialized and PEFT LoRA configured successfully.')


In [ ]:
# ==============================================================================
# CELL 3: SFT TRAINING WITH PREFIX-DELTA ASSISTANT LOSS MASKING
# ==============================================================================
import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# Define prefix-delta masking data processor:
# Tokens belonging to prefix_text (system prompt, user prompt, preceding history) receive -100 in labels.
# Tokens belonging to completion_text (assistant thoughts + tool calls) receive target token IDs for cross-entropy loss.
def prepare_prefix_delta_dataset(jsonl_path, tokenizer, max_seq_length=3072):
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        rows = [json.loads(line) for line in f if line.strip()]
    
    input_ids_list = []
    labels_list = []
    attention_mask_list = []
    
    for r in rows:
        p_text = r['prefix_text']
        f_text = r['text']
        
        p_ids = tokenizer.encode(p_text, add_special_tokens=False)
        f_ids = tokenizer.encode(f_text, add_special_tokens=False)
        
        if len(f_ids) > max_seq_length:
            f_ids = f_ids[:max_seq_length]
            p_ids = p_ids[:min(len(p_ids), max_seq_length)]
            
        labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
        att_mask = [1] * len(f_ids)
        
        input_ids_list.append(f_ids)
        labels_list.append(labels)
        attention_mask_list.append(att_mask)
        
    return Dataset.from_dict({
        'input_ids': input_ids_list,
        'labels': labels_list,
        'attention_mask': attention_mask_list
    })

print('Tokenizing and applying prefix-delta loss masking to train and val sets...')
train_dataset = prepare_prefix_delta_dataset(train_path, tokenizer, max_seq_length=max_seq_length)
val_dataset = prepare_prefix_delta_dataset(val_path, tokenizer, max_seq_length=max_seq_length)
print(f'Prefix-delta datasets ready: train={len(train_dataset)}, val={len(val_dataset)}')

OUTPUT_DIR = WORKING_DIR / 'training_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# TrainingArguments:
# 1. eval_strategy='no': Disables Step-5 evaluation to avoid materializing [1, 3072, 262144] FP32 logits (+4.9 GiB VRAM OOM).
# 2. neftune_noise_alpha=None: Removed NEFTune noise injection to protect <|"|> tool JSON syntax.
# 3. learning_rate=1.0e-4, warmup_ratio=0.1, optim='adamw_8bit'.
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_ratio=0.1,
    num_train_epochs=1,
    learning_rate=1.0e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=2,
    eval_strategy='no',
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    neftune_noise_alpha=None,
    max_grad_norm=1.0,
    seed=42,
    report_to='none',
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors='pt', padding=True),
)

print('Starting Unsloth Gemma 4 LoRA fine-tuning with prefix-delta loss masking...')
train_stats = trainer.train()
print('\nTraining complete! Train stats:')
print(train_stats)


In [ ]:
# ==============================================================================
# CELL 4: EXPORT LORA ADAPTER WEIGHTS & BUILD SUBMISSION PACKAGE
# ==============================================================================
import shutil
from pathlib import Path

ADAPTER_DIR = WORKING_DIR / 'submission' / 'adapters' / 'main_lora'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

print(f'Exporting LoRA adapter weights to {ADAPTER_DIR}...')
# Save ONLY the model PEFT adapter (do NOT save tokenizer to avoid ~15 MB redundant vocab bloat)
model.save_pretrained(str(ADAPTER_DIR))

print('\n================ EXPORTED ADAPTER FILES ================')
total_size = 0
for p in sorted(ADAPTER_DIR.iterdir()):
    if p.is_file():
        sz = p.stat().st_size
        total_size += sz
        print(f'  - {p.name} ({sz / (1024 * 1024):.2f} MB)')
print(f'Total adapter size: {total_size / (1024 * 1024):.2f} MB')
assert (ADAPTER_DIR / 'adapter_config.json').exists(), 'Missing adapter_config.json!'
assert (ADAPTER_DIR / 'adapter_model.safetensors').exists() or (ADAPTER_DIR / 'adapter_model.bin').exists(), 'Missing weights!'

# Verify adapter size constraint (< 100 MB for git/kaggle compatibility)
assert total_size < 100 * 1024 * 1024, f'Adapter size ({total_size / (1024*1024):.2f} MB) exceeds 100 MB limit!'

# Assemble Turnkey Track 2 Submission Package from template repository
SUBMISSION_DIR = WORKING_DIR / 'submission'
TEMPLATE_DIR = (
    Path('/kaggle/input/gemma4-template/submissions/track1_live')
    if Path('/kaggle/input/gemma4-template/submissions/track1_live').exists()
    else Path('submissions/track1_live')
)

if TEMPLATE_DIR.exists():
    print(f'\nAssembling turnkey Track 2 submission using {TEMPLATE_DIR}...')
    # 1. Copy prompts, configs, and skills
    for subdir in ['prompts', 'configs', 'skills']:
        src = TEMPLATE_DIR / subdir
        dst = SUBMISSION_DIR / subdir
        if src.exists():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f'  - Copied {subdir}/')
    
    # 2. Copy eval_config.yaml
    if (TEMPLATE_DIR / 'eval_config.yaml').exists():
        shutil.copy2(TEMPLATE_DIR / 'eval_config.yaml', SUBMISSION_DIR / 'eval_config.yaml')
        print('  - Copied eval_config.yaml')
        
    # 3. Mount adapter in agent.yaml
    agent_yaml = (TEMPLATE_DIR / 'agent.yaml').read_text(encoding='utf-8')
    if 'adapter:' not in agent_yaml and 'adapter_path:' not in agent_yaml:
        agent_yaml += '\nadapter: adapters/main_lora\n'
    (SUBMISSION_DIR / 'agent.yaml').write_text(agent_yaml, encoding='utf-8')
    print('  - Configured agent.yaml with adapter: adapters/main_lora')
else:
    # Fallback: write verified sampling.yaml (preserving full thinking capabilities)
    CONFIGS_DIR = SUBMISSION_DIR / 'configs'
    CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
    sampling_yaml = """temperature: 0.15
top_p: 0.9
max_output_tokens: 4096
thinking_config:
  thinking_budget: 2048
  include_thoughts: true
"""
    (CONFIGS_DIR / 'sampling.yaml').write_text(sampling_yaml, encoding='utf-8')
    print('Configured submission/configs/sampling.yaml with verified settings.')

print('\nTrack 2 LoRA adapter export and submission assembly completed successfully!')
